# 12 · Right targets and run stores on real systems

This notebook drives the Right-side backends of `dtrack_right` against the real Databricks table of one pair and a real S3 scratch prefix. `SparkTarget` (relation SQL, `DESCRIBE QUERY` preflight, grouped partition counts, sampling predicate, one-query statistics), `WarehouseTarget` and `CsvAnchorTarget` all read the pair's real Right table. `S3Store` / `VolumeStore` / `open_store` / `resolve_run` are exercised with real payloads under a throw-away prefix.

It mirrors `tests/unit/dtrack_right/backends/test_targets.py` and `tests/unit/dtrack_right/backends/test_store.py`, with fakes (FakeSpark, moto, RecordingClient) replaced by the SQL warehouse and real S3.

## How to approach

- **Prerequisites**: kernel = the repo `.venv`; `uv sync --extra dbx` (databricks-sdk), plus `--extra athena` / `--extra ldap` if the pair's Left side needs it (section 5 counts the Left side). A `~/.databrickscfg` profile (`DTRACK_DBX_PROFILE`) and a SQL warehouse id (`DTRACK_DBX_WAREHOUSE_ID`). AWS credentials that can write to `DTRACK_PROD_S3` (default boto3 chain: `AWS_PROFILE` or the `.env`). Left credentials `<macro>_USR/_PWD/_DSN` in the project `.env`.
- **Run order**: top to bottom. Sections 1-4 only read Databricks metadata and small samples (fast). Sections 5-9 count the Left side and pull ONE window of the Right table to `WORKDIR` (keep `DAYS` small). Sections 10-13 write only under `S3_SCRATCH/nb12-<tag>/` and `WORKDIR`. Run **Cleanup** at the end.
- **What to look at**: the `col_map` case check in section 2, the SQL-vs-Python rule tables in section 4 (every row should say `True`), and the three-way statistics comparison in sections 7-9 (in-warehouse vs pulled-locally vs CSV anchor).
- **Failure modes**: `preflight ok=False` with `date_col ... not in [...]` = wrong `date_col` or case; `statement FAILED: TABLE_OR_VIEW_NOT_FOUND` = wrong catalog/schema/table or no grant; external-link download errors = presigned URLs blocked, set `DTRACK_LOCAL_DISPOSITION=inline`; `AccessDenied` on S3 = the scratch prefix is not writable.

In [ ]:
import os
from pathlib import Path

CONFIG_PATH = os.environ.get("DTRACK_PROD_CONFIG", "C:/work/dtrack/c.json")        # real pairs config
PAIR = os.environ.get("DTRACK_NB_PAIR", "orders")                                   # pair to inspect
TO_DATE = None                                                                      # None = yesterday; or "YYYY-MM-DD"
DAYS = 3                                                                            # window = last DAYS days up to TO_DATE
S3_SCRATCH = os.environ.get("DTRACK_PROD_S3", "s3://<bucket>/dtrack/nb-scratch")    # only S3 prefix written to
DBX_PROFILE = os.environ.get("DTRACK_DBX_PROFILE", "corp")                          # ~/.databrickscfg profile
WAREHOUSE_ID = os.environ.get("DTRACK_DBX_WAREHOUSE_ID", "")                        # SQL warehouse id
DISPOSITION = os.environ.get("DTRACK_LOCAL_DISPOSITION", "external_links")          # or "inline" if downloads are blocked
WORKDIR = Path.home() / ".local/plans/nbs/_work/12"                                 # local scratch folder
SAMPLE_VALUES = 20                                                                  # distinct values per column for the rule checks
FIXED_KEYS = 5                                                                      # real keys used for the fixed-sample predicate
RUNS_ROOT = os.environ.get("DTRACK_NB_RUNS_ROOT", "")                               # optional existing runs root to list (read-only)
ACCESS_PROBE_URI = ""                                                               # optional s3:// you cannot read, to see exists() raise
KEEP = False                                                                        # True = keep scratch objects and WORKDIR

In [ ]:
# Load the project .env first: the Left credentials (<macro>_USR / <macro>_PWD / <macro>_DSN)
# and the AWS / Databricks defaults live there. DTRACK_ENV_FILE overrides the search.
import copy
import json
import os
import uuid
from datetime import date, timedelta
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv

from dtrack_left.cli.config import load_config

load_dotenv(os.environ.get("DTRACK_ENV_FILE") or find_dotenv(usecwd=True), override=False)

# The real pairs config, validated exactly as `dtrack` validates it.
config = load_config(CONFIG_PATH)
print("pairs in config:", sorted(config["pairs"]))

# The check window: the last DAYS days up to TO (yesterday by default), like the prod smoke plan.
TO = TO_DATE or (date.today() - timedelta(days=1)).isoformat()
FROM = (date.fromisoformat(TO) - timedelta(days=DAYS - 1)).isoformat()

# A private copy of the pair with its date range narrowed to the window; the config itself is untouched.
pair = copy.deepcopy(config["pairs"][PAIR])
pair["fromDate"], pair["toDate"] = FROM, TO
right_cfg = pair["right"]
TAG = f"{PAIR}-{uuid.uuid4().hex[:8]}"
WORKDIR.mkdir(parents=True, exist_ok=True)
print(f"pair {PAIR}: window {FROM}..{TO}, scratch tag {TAG}")
print("right side:", json.dumps(right_cfg, indent=2, default=str))

SCRATCH_NAME = f"nb12-{TAG}"

## Connect to the SQL warehouse

`SparkTarget` only needs an object with `.sql(text).collect()`. The adapter below sends each statement to the real SQL warehouse through `dtrack_local.Warehouse`, so the SQL that `SparkTarget` generates (the same SQL a Databricks job would run with `spark.sql`) is executed unchanged.

In [ ]:
# One WorkspaceClient from ~/.databrickscfg and one Warehouse (Statement Execution API) on top of it.
# Nothing here needs Databricks Connect: every Right-side query goes through the SQL warehouse.
from dtrack_left.backends.databricks import workspace
from dtrack_local.warehouse import Warehouse

client = workspace(DBX_PROFILE)
warehouse = Warehouse(client, WAREHOUSE_ID, disposition=DISPOSITION)
print("signed in as", client.current_user.me().user_name, "on", client.config.host)
print("warehouse", WAREHOUSE_ID, "disposition", DISPOSITION, "->", warehouse.rows("SELECT current_catalog(), current_schema()"))

In [ ]:
# Minimal spark.sql stand-in backed by the real warehouse: rows behave like tuples and have asDict().
# The warehouse returns JSON_ARRAY values as text; SparkTarget already int()/float()s what it reads.
class WarehouseRow(tuple):
    def __new__(cls, names, values):
        row = super().__new__(cls, values)
        row.names = names
        return row

    def asDict(self):
        return dict(zip(self.names, self))


class WarehouseResult:
    def __init__(self, rows):
        self.rows = rows

    def collect(self):
        return self.rows


class WarehouseSpark:
    def __init__(self, warehouse):
        self.warehouse = warehouse
        self.statements = []

    def sql(self, text):
        self.statements.append(text)
        response = self.warehouse.execute(text)
        names = [column.name for column in response.manifest.schema.columns]
        rows = [WarehouseRow(names, row) for chunk in self.warehouse.chunks(response) for row in chunk]
        return WarehouseResult(rows)


spark = WarehouseSpark(warehouse)

## 1. The relation a target reads

`SparkTarget.relation()` turns the Right config into a FROM clause: a quoted `catalog.schema.table`, a `(query) dtrack_source`, or `read_files(...)` for Volume files. A `conn_macro` like `cat.sch` fills catalog/schema when they are not set.

In [ ]:
from dtrack_right.backends.targets import CsvAnchorTarget, SparkTarget

# The real pair's relation and its extra WHERE (config "where"), as every Right query will use them.
target = SparkTarget(spark, right_cfg)
print("relation :", target.relation())
print("where    :", target._where(None))
print("date_col :", right_cfg["date_col"], "date_type:", right_cfg.get("date_type"))

# Labelled illustration only (literal configs, not your pair): the other relation shapes the tests cover.
examples = {
    "query": {"query": "SELECT * FROM main.sales.orders WHERE region = 'EU'"},
    "volume_raw parquet": {"kind": "volume_raw", "path": "/Volumes/c/s/v/orders/"},
    "files csv + {src} patch": {
        "kind": "files",
        "path": "/Volumes/c/s/work/inputs/orders/",
        "format": "csv",
        "options": {"header": "true", "delimiter": "|"},
        "schema_hints": "amount DECIMAL(18,2)",
        "query": "SELECT *, to_date(d, 'dd/MM/yyyy') AS dt FROM {src}",
    },
    "legacy conn_macro": {"conn_macro": "cat.sch", "table": "t"},
}
for label, example in examples.items():
    print(f"[example] {label:24} -> {SparkTarget(None, example).relation()}")

## 2. Preflight: DESCRIBE QUERY and the col_map spelling

`preflight()` runs one `DESCRIBE QUERY SELECT * FROM <relation>` and fails when `date_col` is missing. The column types it records decide numeric vs categorical statistics later, so a `col_map` right name with the wrong case silently becomes categorical (known bug): the check below prints the exact spelling to use.

In [ ]:
# The real DESCRIBE, its column types, and the statement that produced them.
health = target.preflight()
print({k: v for k, v in health.items() if k != "columns"})
print("statement:", spark.statements[-1])
types = pd.Series(getattr(target, "column_types", {}), name="type").to_frame()
display(types)

# Every col_map right name (and the right date_col) must match DESCRIBE exactly, including case.
actual = set(health.get("columns") or [])
wanted = [right_cfg["date_col"], *pair["col_map"].values()]
wrong = {w: next((a for a in actual if a.lower() == w.lower()), "missing") for w in wanted if w not in actual}
print("col_map right names to fix:", wrong or "none")

# A deliberately wrong date_col shows the failure shape preflight returns (no exception raised).
print(SparkTarget(spark, {**right_cfg, "date_col": "dtrack_no_such_column"}).preflight())

## 3. Partition counts in one grouped query

`partition_counts(from, to)` is one `GROUP BY` on the day expression, with the window bounds and the configured `where` combined. Days come back canonical (`YYYY-MM-DD`).

In [ ]:
# Right-side counts for the window, plus the exact SQL that ran.
right_counts = target.partition_counts(FROM, TO)
print("SQL:", spark.statements[-1])
display(pd.Series(right_counts, name="right_rows").sort_index().to_frame())
print(f"{len(right_counts)} day(s), {sum(right_counts.values())} row(s)")

## 4. The normalisation rules: Spark SQL vs Python on real values

`_collapse_decimal_sql` is the Spark twin of `collapse_decimal` (pure decimals lose trailing zeros; `007` and `1e3` stay), and `_norm_sql` mirrors `normalize_value` per key family. Here both run on real distinct values from the window and are compared row by row.

In [ ]:
from dtrack_right.backends.targets import NUMERIC_TYPES, _collapse_decimal_sql, _norm_sql
from dtrack_right.common.dates import bounds_where, quote_ident
from dtrack_right.common.hashing import collapse_decimal, normalize_value

window_where = target._where(
    bounds_where(right_cfg["date_col"], dialect="spark", date_type=right_cfg.get("date_type"), from_date=FROM, to_date=TO)
)

# For every mapped right column: real distinct text values, the SQL collapse result, the Python one.
frames = []
for column in pair["col_map"].values():
    if column not in actual:
        continue
    ref = quote_ident(column, "spark")
    sql = (
        f"SELECT v, {_collapse_decimal_sql('v')} FROM (SELECT DISTINCT trim(cast({ref} as string)) AS v "
        f"FROM {target.relation()} WHERE {window_where} AND {ref} IS NOT NULL LIMIT {SAMPLE_VALUES})"
    )
    rows = warehouse.rows(sql)
    frames.append(pd.DataFrame(
        [(column, v, s, collapse_decimal(v), s == collapse_decimal(v)) for v, s in rows],
        columns=["column", "value", "spark", "python", "same"],
    ))
rules = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
print("collapse rule disagreements:", int((~rules["same"]).sum()) if len(rules) else "no values")
display(rules[rules["value"] != rules["python"]].head(50) if len(rules) else rules)

In [ ]:
# Key normalisation per family (string / numeric / date) on real keys of the window.
key_cols = [k.strip() for k in str(pair.get("key") or "").split(",") if k.strip()]
right_keys = [pair["col_map"].get(k, k) for k in key_cols]
families = [dict(pair.get("key_families") or {}).get(k, "string") for k in key_cols]
print("keys:", list(zip(key_cols, right_keys, families)) or "no key configured")

frames = []
for column, family in zip(right_keys, families):
    ref = quote_ident(column, "spark")
    sql = (
        f"SELECT cast({ref} as string), {_norm_sql(column, family)} FROM {target.relation()} "
        f"WHERE {window_where} LIMIT {SAMPLE_VALUES}"
    )
    rows = warehouse.rows(sql)
    frames.append(pd.DataFrame(
        [(column, family, raw, spark_norm, normalize_value(raw, family)) for raw, spark_norm in rows],
        columns=["key", "family", "raw", "spark", "python"],
    ).assign(same=lambda f: f["spark"] == f["python"]))
keys_frame = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
print("key normalisation disagreements:", int((~keys_frame["same"]).sum()) if len(keys_frame) else "no keys")
display(keys_frame.head(30))

## 5. A real manifest for the window

The statistics calls need a manifest. It is built the way the pipeline builds it: real Left counts (`source_from_config(...).partition_counts`), reconciled with the Right counts, gated, then `build_manifest`. Only days whose counts match become windows.

In [ ]:
from dtrack_left.backends.sources import source_from_config
from dtrack_right.checks.row_check import evaluate_gate, reconcile_partitions, summarize_partitions
from dtrack_right.common.manifest import build_manifest

# Left counts for the same window (read-only SELECT ... GROUP BY on the Left database).
left_source = source_from_config(pair["left"])
print("left preflight:", left_source.preflight())
left_counts = left_source.partition_counts(FROM, TO)

# Reconcile and gate exactly as Stage 1 does, then derive the manifest.
partitions = reconcile_partitions(left_counts, right_counts, pair.get("date_synonyms"))
document = {"partitions": partitions, "summary": summarize_partitions(partitions), "gate_decision": evaluate_gate(partitions, "auto")}
print("summary:", document["summary"], "gate passed:", document["gate_decision"]["passed"])
display(pd.DataFrame(partitions).T)

manifest = build_manifest(f"nb12-{TAG}", PAIR, pair, document)
print("sample:", manifest["sample"], "vintage:", manifest["vintage"], "strict:", manifest["strict"])
print("columns:", manifest["columns"])
print("windows:", [(w["bucket"], w["partitions"], w["n_left"]) for w in manifest["windows"]])
window = manifest["windows"][0] if manifest["windows"] else None
print("first window:", window)
if window is None:
    raise RuntimeError("no day matched on both sides in the window: widen DAYS or move TO_DATE, then rerun")

## 6. The sampling predicate per mode

`_sample_predicate(manifest, window)` returns `None` for `all` (or no keys), an md5-bucket comparison for `fraction`, and a key `IN (...)` list for `fixed` (`1=0` when the list is empty). Each predicate is counted in the warehouse on the first window.

In [ ]:
from dtrack_right.common.dates import partition_where
from dtrack_right.common.manifest import sample_plan

day_filter = partition_where(right_cfg["date_col"], window["partitions"], dialect="spark", date_type=right_cfg.get("date_type"))

# Real keys of the window, used as the "fixed" key list.
key_tuples = []
if manifest["right_key_cols"]:
    columns = ", ".join(f"cast({quote_ident(k, 'spark')} as string)" for k in manifest["right_key_cols"])
    key_tuples = [list(row) for row in warehouse.rows(f"SELECT {columns} FROM {target.relation()} WHERE {target._where(day_filter)} LIMIT {FIXED_KEYS}")]

plans = {
    "configured": (manifest["sample"], window),
    "all": ({"mode": "all"}, window),
    "fraction 0.1": (sample_plan(0.1), window),
    "fixed (real keys)": ({"mode": "fixed", "count": FIXED_KEYS}, {**window, "key_tuples": key_tuples}),
    "fixed (no keys)": ({"mode": "fixed", "count": FIXED_KEYS}, {**window, "key_tuples": []}),
}
for label, (plan, win) in plans.items():
    predicate = target._sample_predicate({**manifest, "sample": plan}, win)
    where = target._where(day_filter if predicate is None else f"{day_filter} AND ({predicate})")
    rows = warehouse.rows(f"SELECT count(*) FROM {target.relation()} WHERE {where}")[0][0]
    print(f"{label:18} rows={rows:>10}  predicate={(predicate or 'None')[:120]}")

## 7. Statistics in one warehouse query

`SparkTarget.window_statistics` issues a single SELECT with per-column counts, exact `COUNT(DISTINCT)`, the sha2 hash sum, mean/std/min/max and lengths, then maps the row back per column. The strict variant lowercases and collapses text; `col_type_overrides` flip numeric/categorical.

In [ ]:
# The configured manifest, computed entirely inside the warehouse.
spark_stats = target.window_statistics(manifest, window)
query = spark.statements[-1]
print("one query, APPROX used:", "APPROX" in query.upper(), "| length:", len(query))
print(query[:1500], "..." if len(query) > 1500 else "")
spark_frame = pd.DataFrame(spark_stats).T
display(spark_frame)

In [ ]:
# Strict mode on the same window: only text columns should change (lowercased / collapsed values).
strict_stats = target.window_statistics({**manifest, "strict": True}, window)
print("count(DISTINCT lower(" in spark.statements[-1], "<- strict lowercases text in SQL")
display(pd.DataFrame({c: {"n_unique": spark_stats[c]["n_unique"], "n_unique_strict": strict_stats[c]["n_unique"], "hash": spark_stats[c]["hash_sum"], "hash_strict": strict_stats[c]["hash_sum"]} for c in spark_stats}).T)

# Overrides: force the first categorical column numeric and the first numeric column categorical.
kinds = {left: spark_stats[manifest["col_map"][left]]["col_type"] for left in manifest["columns"]}
flip = {}
for wanted_kind, new_kind in (("categorical", "numeric"), ("numeric", "categorical")):
    column = next((c for c, k in kinds.items() if k == wanted_kind and c not in flip), None)
    if column:
        flip[column] = new_kind
print("override used:", flip)
override_stats = target.window_statistics({**manifest, "col_type_overrides": flip}, window)
display(pd.DataFrame({c: {"before": kinds[c], "after": override_stats[manifest["col_map"][c]]["col_type"], "mean_after": override_stats[manifest["col_map"][c]]["mean"]} for c in flip}).T)

## 8. WarehouseTarget: pull the window, compute statistics locally

`WarehouseTarget` reuses SparkTarget's relation, preflight and counts, but pulls the window as text (`CAST AS STRING`) to a CSV and runs the Left side's pandas statistics. Comparing it with section 7 through `compare_statistics` shows whether both routes agree on this real table.

In [ ]:
from dtrack_local.pipeline import WarehouseTarget
from dtrack_right.checks.col_check import compare_statistics

# Pull the first window into WORKDIR/right/<pair>/<bucket>/right.csv and summarise it locally.
local_target = WarehouseTarget(warehouse, right_cfg, WORKDIR / "right")
print("preflight:", {k: v for k, v in local_target.preflight().items() if k != "columns"})
sql, selected = local_target.window_sql(manifest, window)
print("pull SQL:", sql[:600])
local_stats = local_target.window_statistics(manifest, window)
pulled_csv = WORKDIR / "right" / PAIR / window["bucket"] / "right.csv"
print("pulled:", pulled_csv, pulled_csv.stat().st_size, "bytes")

# Same comparison the col-check uses: in-warehouse statistics vs local statistics, per column.
rows = [
    compare_statistics(pair=PAIR, window=window["bucket"], column=c, left=spark_stats[c], right=local_stats[c])
    for c in spark_stats
]
display(pd.DataFrame(rows)[["column", "col_type", "verdict", "n_total_diff", "n_unique_diff", "hash_sum_match", "mean_match", "min_match", "max_match"]])

## 9. CsvAnchorTarget on the pulled window

`CsvAnchorTarget` is the local stand-in for the Right side: one `<table>.csv` per table. Feeding it the CSV pulled in section 8 shows the same contract (preflight, counts, statistics) on real rows, and its statistics should equal section 8's.

In [ ]:
import shutil

# Lay the pulled window out as <root>/<table>.csv, the anchor's expected file name.
anchor_root = WORKDIR / "anchor"
anchor_root.mkdir(parents=True, exist_ok=True)
shutil.copyfile(pulled_csv, anchor_root / f"{right_cfg.get('table', 'right')}.csv")
anchor = CsvAnchorTarget(right_cfg, anchor_root, WORKDIR / "anchor_work")
print("preflight:", anchor.preflight())
print("counts   :", anchor.partition_counts(FROM, TO), "vs warehouse", {d: right_counts.get(d) for d in window["partitions"]})

anchor_stats = anchor.window_statistics(manifest, window)
rows = [compare_statistics(pair=PAIR, window=window["bucket"], column=c, left=local_stats[c], right=anchor_stats[c]) for c in local_stats]
display(pd.DataFrame(rows)[["column", "col_type", "verdict", "n_total_left", "n_total_right", "hash_sum_match"]])

# A missing anchor file and a missing date column, as preflight reports them.
print(CsvAnchorTarget({"table": "dtrack_missing", "date_col": "d"}, anchor_root, WORKDIR).preflight())
print(CsvAnchorTarget({**right_cfg, "date_col": "dtrack_no_such_column"}, anchor_root, WORKDIR).preflight())

## 10. S3Store round trip on the scratch prefix

`S3Store` is the run folder on S3: JSON and bytes in, listings out. Everything below lands under `S3_SCRATCH/nb12-<tag>/` and is deleted in Cleanup. The payloads are this notebook's real manifest and statistics.

In [ ]:
from dtrack_right.backends.store import S3Location, S3Store, VolumeStore, open_store, resolve_run, run_is_complete

scratch = f"{S3_SCRATCH.rstrip('/')}/{SCRATCH_NAME}"
print(S3Location.parse(scratch))
store = S3Store(f"{scratch}/store")

# JSON and bytes: the real manifest and the pulled window CSV.
print(store.put_json("manifest/probe.json", manifest))
print("round trip equal:", store.get_json("manifest/probe.json") == json.loads(json.dumps(manifest)))
store.put_bytes("data/right.csv", pulled_csv.read_bytes(), content_type="text/csv")
print("bytes equal:", store.get_bytes("data/right.csv") == pulled_csv.read_bytes())
print("exists:", store.exists("manifest/probe.json"), store.exists("manifest/nope.json"))
print("list_keys(''):", store.list_keys(""))
print("list_prefixes():", store.list_prefixes())

# Databricks mkdir on an S3-backed Volume leaves 0-byte "folder/" objects; list_keys must skip them.
store.client.put_object(Bucket=store.bucket, Key=store._key("m/"), Body=b"")
store.put_json("m/a.json", {"ok": True})
print("list_keys('m/') skips the marker:", store.list_keys("m/"))

## 11. The unit protocol: commit, heartbeat, results, dead, close

`commit_unit` writes the payload first and the `_INDEX/` pointer last; `ready_unclaimed` = pointers minus results/dead. The payloads here are the real local statistics per column of the first window.

In [ ]:
from dtrack_right.common.manifest import unit_id

run_store = S3Store(f"{scratch}/runs/protocol")
for sequence, column in enumerate(manifest["columns"], start=1):
    uid = unit_id(PAIR, window["bucket"], column)
    pointer = {"pair": PAIR, "window": window["bucket"], "column": column, "right_column": manifest["col_map"][column]}
    print(run_store.commit_unit(sequence, uid, local_stats[manifest["col_map"][column]], pointer))
run_store.heartbeat()
print("heartbeat age (s):", run_store.heartbeat_age())
print("ready_unclaimed:", run_store.ready_unclaimed())

# Answer the first unit with a real comparison and mark the second one dead (labelled demo failure).
first, *rest = [pointer for _key, pointer in run_store.list_json("_INDEX/")]
left = run_store.get_json(first["payload_key"])
result = compare_statistics(pair=PAIR, window=first["window"], column=first["column"], left=left, right=spark_stats[first["right_column"]])
run_store.put_result(first["unit_id"], {**result, "unit_id": first["unit_id"], "right_col": first["right_column"]})
if rest:
    run_store.put_dead(rest[0]["unit_id"], {"unit_id": rest[0]["unit_id"], "error": "demo: marked dead by notebook 12"})
print("ready_unclaimed after:", run_store.ready_unclaimed())
print("list_json start_after first:", [k for k, _ in run_store.list_json("_INDEX/", start_after=run_store.list_keys("_INDEX/")[0])])
run_store.close_manifests([PAIR], manifest["expected_units"])
run_store.close_row_check([PAIR])
print(run_store.list_keys(""))

## 12. VolumeStore: the same contract on a directory

`VolumeStore` is what a Unity Catalog Volume path or a local folder uses: writes go to `<file>.part` then rename, and listings hide `.part` files. Same calls as section 11, under `WORKDIR`.

In [ ]:
volume = VolumeStore(WORKDIR / "volume" / "protocol")
for sequence, column in enumerate(manifest["columns"], start=1):
    uid = unit_id(PAIR, window["bucket"], column)
    volume.commit_unit(sequence, uid, local_stats[manifest["col_map"][column]], {"pair": PAIR, "window": window["bucket"], "column": column})
volume.heartbeat()
print("keys:", volume.list_keys("_INDEX/"))
print("ready_unclaimed:", volume.ready_unclaimed(), "heartbeat age:", volume.heartbeat_age())

# A half-written file stays invisible; prefixes with a leading slash or a partial name still match.
(volume.root / "_INDEX" / "99999999-half.json.part").write_text("{")
print("with .part present:", volume.list_keys("/_INDEX/")[-1:], "| partial prefix:", len(volume.list_keys("_INDEX/0000000")))
print("missing folder:", volume.list_keys("nope/deeper/"))
print("S3 and Volume list the same relative keys:", sorted(volume.list_keys("units/")) == sorted(k for k in run_store.list_keys("units/")))

## 13. open_store, resolve_run and run_is_complete

`open_store` picks S3 or directory from the location string. `resolve_run` returns an explicit run, a location that already is a run, or the newest unfinished run under a root (`(None, None)` when every run is complete). The same three-run layout is built on both backends.

In [ ]:
roots = {"s3": f"{scratch}/resolve", "volume": str(WORKDIR / "volume" / "resolve")}
for backend, runs_root in roots.items():
    root = open_store(runs_root)
    print(f"--- {backend}: {type(root).__name__} at {runs_root}")
    root.child("20260901T000000Z-aaa").put_json("_STATUS.json", {"state": "complete"})
    root.child("20260902T000000Z-bbb").put_json(f"row_check/{PAIR}.left.json", {"pair": PAIR, "counts": left_counts})
    root.child("20260903T000000Z-ccc").put_json(f"row_check/{PAIR}.left.json", {"pair": PAIR, "counts": left_counts})
    print("prefixes:", root.list_prefixes())
    found, run_id = resolve_run(runs_root)
    print("newest unfinished:", run_id, found.exists(f"row_check/{PAIR}.left.json"))
    print("explicit:", resolve_run(runs_root, "20260902T000000Z-bbb")[1])
    print("run location itself:", resolve_run(f"{runs_root}/20260902T000000Z-bbb/")[1])
    print("complete?", {name: run_is_complete(root.child(name)) for name in root.list_prefixes()})
    done = open_store(runs_root + "-done")
    done.child("r1").put_json("_STATUS.json", {"state": "complete"})
    print("all complete ->", resolve_run(runs_root + "-done"))

# Optional, read-only: what resolve_run would pick under your real runs root.
if RUNS_ROOT:
    real_root = open_store(RUNS_ROOT)
    print("real runs:", real_root.list_prefixes()[-10:])
    print("resolve_run picks:", resolve_run(RUNS_ROOT)[1])

# Optional: exists() returns False for a missing key but raises on access errors (e.g. 403).
if ACCESS_PROBE_URI:
    try:
        print(S3Store(ACCESS_PROBE_URI).exists("x"))
    except Exception as exc:
        print("raised as expected:", type(exc).__name__, exc)

## Cleanup

Deletes `S3_SCRATCH/nb12-<tag>/` and `WORKDIR` unless `KEEP = True`.

In [ ]:
# Delete everything this notebook wrote under the scratch prefix (and the local WORKDIR) unless KEEP.
import shutil

import boto3

scratch = f"{S3_SCRATCH.rstrip('/')}/{SCRATCH_NAME}"
if KEEP:
    print("KEEP=True: left", scratch, "and", WORKDIR)
else:
    bucket, _, prefix = scratch.removeprefix("s3://").partition("/")
    s3 = boto3.client("s3", endpoint_url=os.getenv("DTRACK_S3_ENDPOINT_URL") or None)
    deleted = 0
    for page in s3.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefix + "/"):
        keys = [{"Key": row["Key"]} for row in page.get("Contents") or []]
        if keys:
            s3.delete_objects(Bucket=bucket, Delete={"Objects": keys})
            deleted += len(keys)
    shutil.rmtree(WORKDIR, ignore_errors=True)
    print(f"deleted {deleted} object(s) under {scratch} and {WORKDIR}")

## What to check

- Section 2: `col_map right names to fix: none`; otherwise copy the spelling shown into the pair config.
- Section 4: zero rule disagreements between Spark SQL and Python (collapse and key normalisation).
- Section 5: the gate passed and there is at least one window; otherwise counts differ on every day.
- Sections 7-9: verdicts `match` between in-warehouse, pulled-local and CSV-anchor statistics; strict and override variants change only what they should; no `APPROX` in the query.
- Sections 10-13: round trips equal, folder marker skipped, `ready_unclaimed` drops after result/dead, `resolve_run` picks `...-ccc` and returns `(None, None)` when all runs are complete.